In [9]:
import collections
from glob import glob
import joblib
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import os
import numpy as np
import torch.nn as nn
import torch.utils.data
from torchvision import transforms
from PIL import ImageFile
import torchvision.models as models
from PIL import Image, ImageFile
import scanpy as sc

ImageFile.LOAD_TRUNCATED_IMAGES = True


class PTC_cell(torch.utils.data.Dataset):
    def __init__(self, root, slide='11_breast_cancer3',transform=None, stain_norm=False):
        super(PTC_cell, self).__init__()
        self.root = root
        self.slide = slide
        self.transform = transform
        self.stain_norm = stain_norm

        patch_path = os.path.join(root, slide, 'patches')
        patch = os.listdir(patch_path)
        patch_list = [x.split('.')[0] for x in patch]

        cell_label = pd.read_csv(os.path.join(root, slide, 'cell_ratio_from_raw70m.csv'), index_col=0)
        # cell_label = cell_label.apply(lambda x: x*100.0)
        
        gene_label = pd.read_csv(os.path.join(root, slide, 'high_250_stdata.csv'), index_col=0)
        label_df = pd.merge(gene_label, cell_label, left_index=True, right_index=True)

        label_index_set = set(label_df.index)
        patch_index_set = set(patch_list)
        and_set = label_index_set & patch_index_set

        patch_list = list(and_set)
        self.label_df = label_df.loc[patch_list]
        self.patch = patch_list


    def __getitem__(self, index):
        patch_id = self.patch[index]
        patch_path = os.path.join(self.root, self.slide, 'patches', patch_id)
        patch = Image.open(patch_path+'.jpg').convert('RGB')
        data = transforms.Resize((224, 224))(patch)
        if self.transform is not None:
            data = self.transform(data)

        label = self.label_df.loc[patch_id].values
        label = torch.Tensor(label)

        return patch_id, data, label

    def __len__(self):
        return len(self.patch)


class fully_connected(nn.Module):
    def __init__(self, model, num_ftrs, num_classes):
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        out_1 = x
        out_3 = self.fc_4(x)
        out_3 = torch.relu(out_3)
        return out_1, out_3

In [10]:
# prepare necessary arguments and WSI sample list

tif_list = '/data1/r20user3/shared_project/Hist2Cell/data/stnet/'
tif_list = glob(tif_list + '/*[!xlsx|ipynb]')
tif_list.sort()
tif_list

['/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_D1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23268_C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23268_C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23268_D1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23269_C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23269_C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23269_D1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23270_D2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23270_E1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23270_E2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23272_D2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23272_E1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23272_E2',
 '/data1/r20user3/shared_

In [11]:
# define test slides list

test_slides = list()
for tif in tif_list:
    tif_path = tif.split('/')[-1]
    test_slides.append(tif_path)
test_slides

['23209_C1',
 '23209_C2',
 '23209_D1',
 '23268_C1',
 '23268_C2',
 '23268_D1',
 '23269_C1',
 '23269_C2',
 '23269_D1',
 '23270_D2',
 '23270_E1',
 '23270_E2',
 '23272_D2',
 '23272_E1',
 '23272_E2',
 '23277_D2',
 '23277_E1',
 '23277_E2',
 '23287_C1',
 '23287_C2',
 '23287_D1',
 '23288_D2',
 '23288_E1',
 '23288_E2',
 '23377_C1',
 '23377_C2',
 '23377_D1',
 '23450_D2',
 '23450_E1',
 '23450_E2',
 '23506_C1',
 '23506_C2',
 '23506_D1',
 '23508_D2',
 '23508_E1',
 '23508_E2',
 '23567_D2',
 '23567_E1',
 '23567_E2',
 '23803_D2',
 '23803_E1',
 '23803_E2',
 '23810_D2',
 '23810_E1',
 '23810_E2',
 '23895_C1',
 '23895_C2',
 '23895_D1',
 '23901_C2',
 '23901_D1',
 '23903_C1',
 '23903_C2',
 '23903_D1',
 '23944_D2',
 '23944_E1',
 '23944_E2',
 '24044_D2',
 '24044_E1',
 '24044_E2',
 '24105_C1',
 '24105_C2',
 '24105_D1',
 '24220_D2',
 '24220_E1',
 '24220_E2',
 '24223_D2',
 '24223_E1',
 '24223_E2']

In [12]:
# prepare my GPU

gpu_list = [0]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [13]:
# prepare the feature extractor model - KimiaNet

model = models.densenet121(pretrained=True)
model.features = nn.Sequential(model.features, nn.AdaptiveAvgPool2d(output_size=(1, 1)))
num_ftrs = model.classifier.in_features
model_final = fully_connected(model.features, num_ftrs, 250+39)

KimiaNetPyTorchWeights_path = "/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/KimiaNetPyTorchWeights.pth"
# Checkpoint_path = '/data2/r10user3/Spatial_Gene_Cell_Ratio/code/model_weights/densenet_multi_cell_gene_epoch500_best_cell_average.pth'

checkpoint = torch.load(KimiaNetPyTorchWeights_path)
# checkpoint = torch.load(Checkpoint_path)

new_state_dict = collections.OrderedDict()
for k, v in checkpoint.items():
    if 'fc_4' in k:
        continue
    name = k[7:]  # remove "module."
    new_state_dict[name] = v
model2_dict = model_final.state_dict()
state_dict = {k:v for k,v in new_state_dict.items() if k in model2_dict.keys()}
model2_dict.update(state_dict)
model_final.load_state_dict(model2_dict)
model = model_final
model = model.to(device)
model

fully_connected(
  (model): Sequential(
    (0): Sequential(
      (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu0): ReLU(inplace=True)
      (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (denseblock1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu2): ReLU(inplace=True)
          (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (denselayer2): _DenseLayer(
          (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.

In [14]:
# define test dataset transform

test_transform_pcam = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [15]:
batch_size = 256
patch_dir = "/home/r15user3/Documents/shared_project/Hist2Cell/data/stnet"

graphs = dict()
for slide in test_slides:
    print(slide)
    test_data = PTC_cell(root=patch_dir, slide=slide,transform=test_transform_pcam)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4)

    spots_coord = pd.read_csv(os.path.join("/home/r15user3/Documents/shared_project/Hist2Cell/data/stnet", slide, "spots.csv"), index_col=0)

    with torch.no_grad():
        # model.eval()
        test_prediction_array = []
        test_label_array = []
        test_name_array = []

        for name, data, label in test_loader:
            test_name_array.append(list(name))
            data = data.to(device)
            label = label.to(device)
            label = label.float()
            label = label.squeeze()
            test_label_array.append(label.cpu().detach().numpy())
            
            features, output = model(data)
            test_prediction_array.append(features.squeeze().cpu().detach().numpy())
            # test_prediction_array.append(data.cpu().detach().numpy())

    for i in range(len(test_prediction_array)):
        if len(test_prediction_array[i].shape) <= 1:
            test_prediction_array[i] = test_prediction_array[i][np.newaxis, :]
    for i in range(len(test_label_array)):
        if len(test_label_array[i].shape) <= 1:
            test_label_array[i] = test_label_array[i][np.newaxis, :]

    test_prediction_array = np.concatenate(test_prediction_array)
    test_label_array = np.concatenate(test_label_array)
    
    test_names = list()
    for names in test_name_array:
        test_names = test_names+names
    test_node_x_y = list()
    for item in test_names:
        test_node_x_y.append([int(item.split('x')[0]), int(item.split('x')[1])])

    adj = np.zeros((len(test_node_x_y), len(test_node_x_y)))

    for i in range(len(test_node_x_y)):
        for j in range(len(test_node_x_y)):
            if i == j:
                adj[i][j] = 1.0
            else:
                x1 = test_node_x_y[i][0]
                y1 = test_node_x_y[i][1]
                x2 = test_node_x_y[j][0]
                y2 = test_node_x_y[j][1]

                if x2 <= x1 - 2 or x2 >= x1 + 2 or y2 <= y1 - 2 or y2 >= y1 + 2:
                    continue
                else:
                    adj[i][j] = 1.0

    graphs[slide] = {
        'features': test_prediction_array,
        'labels': test_label_array,
        'adj': adj,
        'names': test_names,
        'coord': spots_coord.loc[test_names].values,
    }

    print("OK")

23209_C1


FileNotFoundError: [Errno 2] No such file or directory: '/home/r15user3/Documents/shared_project/Hist2Cell/data/stnet/23209_C1/patches'

In [ ]:
graphs['23209_C1']['features'].shape

(294, 1024)

In [ ]:
graphs['23209_C1']['coord'].shape

(294, 2)

In [ ]:
graphs['23209_C1']['names']

['12x21',
 '19x14',
 '16x15',
 '17x9',
 '18x11',
 '19x5',
 '13x19',
 '10x15',
 '16x9',
 '13x23',
 '6x12',
 '10x17',
 '20x6',
 '9x12',
 '12x10',
 '18x9',
 '11x23',
 '19x17',
 '8x18',
 '6x16',
 '6x25',
 '8x26',
 '4x20',
 '12x9',
 '13x13',
 '9x15',
 '14x23',
 '4x14',
 '8x13',
 '22x13',
 '17x20',
 '10x21',
 '16x18',
 '5x13',
 '14x15',
 '11x21',
 '19x18',
 '14x9',
 '14x20',
 '6x21',
 '6x13',
 '20x17',
 '11x12',
 '19x19',
 '3x14',
 '7x23',
 '13x20',
 '7x14',
 '11x20',
 '3x13',
 '20x16',
 '19x13',
 '19x9',
 '21x17',
 '21x16',
 '20x13',
 '24x15',
 '14x19',
 '16x16',
 '16x10',
 '7x17',
 '15x22',
 '18x8',
 '6x15',
 '17x17',
 '12x22',
 '22x16',
 '9x25',
 '11x24',
 '23x16',
 '16x20',
 '20x5',
 '16x12',
 '6x24',
 '8x25',
 '3x16',
 '7x15',
 '23x17',
 '13x8',
 '17x19',
 '2x20',
 '8x16',
 '12x24',
 '16x11',
 '15x14',
 '5x21',
 '4x22',
 '5x22',
 '14x22',
 '22x15',
 '15x20',
 '13x17',
 '19x7',
 '11x22',
 '18x20',
 '8x22',
 '21x11',
 '5x19',
 '16x17',
 '18x16',
 '16x8',
 '12x23',
 '14x21',
 '2x15',
 '20x

In [ ]:
graph = graphs
for slide in graph:
    print(slide)

23209_C1
23209_C2
23209_D1
23268_C1
23268_C2
23268_D1
23269_C1
23269_C2
23269_D1
23270_D2
23270_E1
23270_E2
23272_D2
23272_E1
23272_E2
23277_D2
23277_E1
23277_E2
23287_C1
23287_C2
23287_D1
23288_D2
23288_E1
23288_E2
23377_C1
23377_C2
23377_D1
23450_D2
23450_E1
23450_E2
23506_C1
23506_C2
23506_D1
23508_D2
23508_E1
23508_E2
23567_D2
23567_E1
23567_E2
23803_D2
23803_E1
23803_E2
23810_D2
23810_E1
23810_E2
23895_C1
23895_C2
23895_D1
23901_C2
23901_D1
23903_C1
23903_C2
23903_D1
23944_D2
23944_E1
23944_E2
24044_D2
24044_E1
24044_E2
24105_C1
24105_C2
24105_D1
24220_D2
24220_E1
24220_E2
24223_D2
24223_E1
24223_E2


In [ ]:
# save the graphs

save_path='/home/r15user3/Documents/shared_project/Hist2Cell/code/data_preprocessing/extracted_graph/stnet_high250gene_celltype39_KimiaNet_extracted_graphs'
joblib.dump(graphs, save_path)

['/home/r15user3/Documents/shared_project/Hist2Cell/code/data_preprocessing/extracted_graph/stnet_high250gene_celltype39_KimiaNet_extracted_graphs']

In [ ]:
graph = joblib.load(save_path)
graph['23209_C1'].keys()

dict_keys(['features', 'labels', 'adj', 'names', 'coord'])

In [ ]:
graph['23209_C1']['features'].shape

(294, 1024)

In [ ]:
graph['23209_C1']['labels'].shape

(294, 289)